### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
import random
import numpy as np
import tensorflow as tf
seed_value = 42
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
random.seed(seed_value)

In [ ]:
embedding_index = {}
import numpy as np
with open('./ Notebooks/Hsoub/L9/cc.ar.300.vec', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = coefs
print("len(embedding_index)", len(embedding_index))

In [ ]:
import pandas as pd
data = pd.read_csv('./ Notebooks/Hsoub/L9/data/Arabic Sentiment Analysis Dataset - SS2030.csv',sep=';')

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')
from nltk.tokenize import word_tokenize

import string

def cleanText(text):
    numbers="0123456789"
    arabic_punctuation='''`÷×؛<>_()*^ـ،/:"؟.,'~¦+|!”…“–ـ'''
    english_punctuation=string.punctuation
    del_chars=english_punctuation+arabic_punctuation+numbers

    for char in del_chars:
        print()
        text = text.replace(char, "")
    text = text.replace('\n', ' ')
    text = text.strip(' ')
    listStopwords = stopwords.words('arabic')
    tokens_list=word_tokenize(text)
    filtered = []
    for txt in tokens_list:
        if txt not in listStopwords:
            filtered.append(txt)
    return filtered
data['text'] = data['text'].apply(cleanText)
texts = data['text'].values
labels = data['Sentiment'].values


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
from keras.preprocessing.sequence import pad_sequences
maxlen = max([len(seq) for seq in sequences])
sequences_padded = pad_sequences(sequences, maxlen=maxlen)
vocab_size = len(tokenizer.word_index )+1


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(sequences_padded, labels, test_size=0.3, random_state=42)
X_test, X_val, y_test, y_val = train_test_split( X_test, y_test, test_size=0.5, random_state=42)

In [ ]:
embedding_dim = 300
embedding_matrix = np.zeros((vocab_size, embedding_dim))

In [ ]:
word_index = tokenizer.word_index
for word,id in word_index.items():
    embedding_vector = embedding_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[id] = embedding_vector

In [ ]:
from keras.layers import Input, Embedding, LSTM, Dense
from keras.models import Sequential

model = Sequential()
model.add(Input (shape=(maxlen,)))
embedding_dim = 300
model.add(Embedding(input_dim= vocab_size,
                    input_length=maxlen,
                    output_dim= embedding_dim,
                    weights=[embedding_matrix],
                    trainable=False
                    ))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
epochs=5
batch_size=16
model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_val, y_val))

In [ ]:
y_pred_probs = model.predict(X_test)

y_pred = (y_pred_probs > 0.5).astype(int)
from sklearn.metrics import accuracy_score
accuracy = round(100*accuracy_score(y_test, y_pred),2)
print(f'Test Accuracy: {accuracy:.2f}')

In [ ]:
from keras.layers import Input, Embedding, LSTM, Dense
from keras.models import Sequential

model = Sequential()
model.add(Input (shape=(maxlen,)))
embedding_dim = 300
model.add(Embedding(input_dim= vocab_size,
                    input_length=maxlen,
                    output_dim= embedding_dim,
                    weights=[embedding_matrix],
                    trainable=True
                    ))
model.add(LSTM(64))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
epochs=5
batch_size=16
model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_val, y_val))

In [ ]:
y_pred_probs = model.predict(X_test)

y_pred = (y_pred_probs > 0.5).astype(int)
from sklearn.metrics import accuracy_score
accuracy = round(100*accuracy_score(y_test, y_pred),2)
print(f'Test Accuracy: {accuracy:.2f}')